Neural Training

In [ ]:
print('--- STEP 0: INSTALLING DEPENDENCIES ---')
# Install compatible versions of the required libraries for CUDA 13
# Remove conflicting versions
!pip uninstall -y transformers trl peft accelerate bitsandbytes datasets

# Install compatible versions
!pip install -q \
transformers==4.41.2 \
trl==0.9.6 \
peft==0.11.1 \
accelerate==0.31.0 \
datasets==2.20.0 \
-q bitsandbytes==0.48.1 triton

In [ ]:
import transformers, trl, peft, accelerate, bitsandbytes, datasets, torch

print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("datasets:", datasets.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
print("--- STEP 1: MERGING DATASETS ---")
# Load the cleaned datasets you created earlier
codex_df = pd.read_csv("codex_clean.csv")
mqa_df = pd.read_csv("unified_mqa_train.csv")

In [ ]:
master_train_df = pd.concat([codex_df, mqa_df], ignore_index=True)
master_train_df = master_train_df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle
dataset = Dataset.from_pandas(master_train_df)


In [ ]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def format_prompt(row):
    """The strict format we force the AI to learn."""
    prompt = f"### Instruction: Write Python code to solve the math problem. Store the answer in 'result'.\n"
    prompt += f"### Question:\n{row['question']}\n"
    prompt += f"### Code:\n{row['code']}"
    return {"text": prompt + tokenizer.eos_token}

formatted_dataset = dataset.map(format_prompt)

In [ ]:
print("--- STEP 2: LOADING MODEL IN 4-BIT ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
# !pip uninstall -y numpy pandas
# !pip install -q numpy==1.26.4 pandas==2.2.2

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

In [ ]:
print("--- STEP 3: APPLYING QLoRA ---")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)


In [ ]:
print("--- STEP 4: TRAINING ---")
training_args = TrainingArguments(
    output_dir="./phi3-math-agent",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=500, # Use max_steps for a quick test, or change to num_train_epochs=1 for full training
    fp16=True,
    optim="paged_adamw_8bit"
)


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()


In [ ]:
print("--- STEP 5: SAVING ---")
trainer.model.save_pretrained("phi3-neuro-symbolic-adapter")
tokenizer.save_pretrained("phi3-neuro-symbolic-adapter")